In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import time
import numpy as np  
import pandas as pd 
import os

import matplotlib.pyplot as plt
import sklearn
import xgboost as xgb
from sklearn import metrics
from sklearn import preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder,MinMaxScaler,StandardScaler
from sklearn.metrics import classification_report,roc_auc_score,average_precision_score,confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from itertools import cycle
from sklearn.metrics import auc,roc_curve,roc_auc_score

from keras.layers import Dense, LSTM, MaxPool1D, Flatten, Dropout # importing dense layer
from keras.layers import Conv1D, Dense, Dropout, Flatten, MaxPooling1D
from keras.models import Sequential #importing Sequential layer
from keras.layers import Input
from keras.models import Model
# representation of model layers
from keras.utils.vis_utils import plot_model

from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import roc_curve

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import Normalizer
from sklearn.preprocessing import LabelBinarizer

import warnings
warnings.filterwarnings("ignore")

In [ ]:

data_train = pd.read_csv('C:/Python/NSL_KDD/V2_traffic_prediction/M_Dataset_2_70.csv', header=None)
data_test = pd.read_csv('C:/Python/NSL_KDD/V2_traffic_prediction/M_Dataset_2_30.csv', header=None)
data= pd.concat([data_train, data_test], ignore_index=True)

In [ ]:
data.head(10)

In [ ]:
data.info()

In [ ]:
data.describe().T

In [ ]:
data.isna().sum()

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
print('?????????? ? data: ',data.duplicated().sum())

In [ ]:
for column in data.columns:
    if data[column].dtype == type(object):
        le = LabelEncoder()
        data[column] = le.fit_transform(data[column])
        
X = data.iloc[:,0:83]
Y = data.iloc[:,83:84]

scaler = Normalizer().fit(X)
trainX = scaler.transform(X)

y_train1 = np.array(Y)

y_data= to_categorical(y_train1)
X_data = np.reshape(trainX, (trainX.shape[0],trainX.shape[1],1))

In [ ]:
X_data.shape

In [ ]:
y_data.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_data,y_data, test_size=0.20, random_state=577)
print(X_train.shape,'\n',X_test.shape)

In [ ]:
model = Sequential()
model.add(Conv1D(64, 3, padding="same",strides=1,activation="relu",input_shape=(83, 1)))
model.add(MaxPooling1D(pool_size=2))
model.add(Flatten())
model.add(Dense(512, activation="relu"))#128
model.add(Dense(256, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(6, activation="softmax"))

model.compile(loss="categorical_crossentropy", optimizer="adam",metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
result= model.fit(X_train, y_train,
                #validation_split=0.30
                validation_data=(X_test,y_test )
                ,epochs=2, batch_size=32) #epochs=20

In [ ]:
y_pred = model.predict(X_test) #probabilty
y_pred = (y_pred > 0.5)

history=result

print(history)

In [ ]:
print(history.history.keys())

In [ ]:
expected = y_test
cmatrix = confusion_matrix(expected.argmax(axis=1), y_pred.argmax(axis=1))
print('Confusion Matrix')
print(cmatrix)

In [ ]:
def accuracy(confusion_matrix):
    diagonal_sum = confusion_matrix.trace()
    sum_of_all_elements = confusion_matrix.sum()
    return diagonal_sum / sum_of_all_elements 

In [ ]:
print("accuracy :" , accuracy(cmatrix))

In [ ]:
print('Classification Report')

print(classification_report(expected, y_pred))

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy',fontsize=25)
plt.ylabel('accuracy',fontsize=25)
plt.xlabel('epoch',fontsize=25)
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss',fontsize=25)
plt.ylabel('loss',fontsize=25)
plt.xlabel('epoch',fontsize=25)
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
lb = LabelBinarizer()
lb.fit(y_test)
y_test = lb.transform(y_test)
y_pred = lb.transform(y_pred)
    
fpr = dict()
tpr = dict()
roc_auc = dict()
    
for i in range(6):
    fpr[i], tpr[i], _ = metrics.roc_curve(y_test[:,i].astype(int), y_pred[:,i])
        
    roc_auc[i] = auc(fpr[i], tpr[i])
    
plt.figure(figsize=(10, 10))
lw = 2
    
plt.plot([0, 1], [0, 1], color='navy', lw=lw, linestyle='--')
target_classes = ['U2R','BFA','DDoS','DoS', 'Probe ','Normal']

plt.title('Receiver Operating Characteristic(ROC) Curve',fontsize=25)
plt.legend(loc="lower right")
colors = cycle(['green', 'yellow', 'blue', 'red', 'orange','black'])
 
for i, color in zip(range(6), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=lw, label= target_classes[i] + " "' (' 'AUC = {1:0.2f} )'''.format(i, roc_auc[i]))
            
    plt.plot([0, 1], [0, 1], color='black', lw=lw, linestyle='--')
    
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.legend(loc = 'lower right')
plt.xlabel('False Positive Rate',fontsize=25)
plt.ylabel('True Positive Rate',fontsize=25)